# Final enhancement review

This notebook audits an existing neural-network + V3 candidate against the exact three-seed submission that scored **0.86380** on Kaggle. The audited candidate has a local diagnostic score of **0.860764**, compared with **0.860009** for the reference. Its Kaggle score remains unknown.

The notebook uses project scripts and saved predictions. It does not retrain models or submit to Kaggle by default.

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display, Code
ROOT=Path.cwd()
display(Code(filename=str(ROOT/'final_ensemble_review.py'),language='python'))

In [ ]:
RUN_AUDIT=False
if RUN_AUDIT:
    import subprocess
    subprocess.run([str(ROOT/'.venv/bin/python'),'final_ensemble_review.py'],cwd=ROOT,check=True)
OUT=ROOT/'outputs/final_review'
display(pd.read_csv(OUT/'comparison.csv'))
display(pd.read_csv(OUT/'gamma_error_slices.csv'))
display(json.loads((OUT/'conditional_intervals.json').read_text()))

## Interpretation

The complete neural-network + V3 blend improves both overall metrics. Missing rapid readings remain the largest error concentration relative to their row count; floor readings worsen slightly. Grouped bootstrap intervals favor the complete candidate, but are conditional on cached predictions. They exclude model-selection and retraining uncertainty.

The inherited V2 regression stack is not fully nested, and the neural-network blend weight was chosen using existing OOF results. This is evidence for a Kaggle trial, not proof of a higher unseen-test score. Keep the successful three-seed submission as a fallback.

In [ ]:
reference=ROOT/'outputs/v2/submission_3seed.csv'
source=ROOT/'outputs/v2/submission_nn_blend_v3.csv'
candidate=OUT/'submission_nn_v3_verified.csv'
assert hashlib.sha256(reference.read_bytes()).hexdigest()=='c49a7788060dd229e1c6072b463c7f90233627f9be63cb4ad59bea4b8d0bd95a'
assert candidate.read_bytes()==source.read_bytes()
frame=pd.read_csv(candidate);sample=pd.read_csv(ROOT/'data/sample_submission.csv')
assert len(frame)==6399 and frame.ID.equals(sample.ID)
assert frame.columns.tolist()==['ID','Smoking','Gamma_GT']
assert np.isfinite(frame[['Smoking','Gamma_GT']]).all().all()
assert frame.Smoking.between(0,1).all() and frame.Gamma_GT.between(1,1000).all()
display(json.loads((OUT/'audit.json').read_text()))
print('Verified candidate:',candidate)